# Tutorial 5 — Measuring it the way the papers do

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/juanmigutierrez/generative-recommendation-engine/blob/main/notebooks/tutorial_05_evaluation.ipynb)

Companion to the two results sections. Here you will:

1. see the two protocols side by side for one user — the same history, the "predict next year" target versus the "predict the next item" target,
2. load the trained SASRec and Semantic-ID checkpoints and reproduce the leave-one-out table on a sample of users (all 94,762 takes ~30 min on a T4; the default is 3,000),
3. compare the four models' lists for any user, with the true next item marked.

Use a GPU runtime; beam search is the slow part.

In [ ]:
#@title Setup — clone the repo, install deps, download the pre-computed artifacts (~1 min)
import os, sys, urllib.request
REPO_URL = "https://github.com/juanmigutierrez/generative-recommendation-engine"
RELEASE  = REPO_URL + "/releases/download/v1.0-artifacts"
QUICK    = os.environ.get("TUTORIAL_QUICK") == "1"   # tiny sizes for headless smoke tests

if not os.path.exists("backend"):
    if not os.path.exists("generative-recommendation-engine"):
        !git clone -q {REPO_URL}
    %cd generative-recommendation-engine
if "google.colab" in sys.modules:
    !pip install -q implicit lightgbm pyarrow ipywidgets 2>&1 | tail -1
os.makedirs("data/processed", exist_ok=True)

def fetch(*names):
    """Download an artifact from the GitHub release unless it is already on disk."""
    for n in names:
        p = os.path.join("data", "processed", n)
        if not os.path.exists(p):
            print("downloading", n, "...")
            urllib.request.urlretrieve(f"{RELEASE}/{n}", p)

fetch("item_catalog.parquet", "user_catalog.parquet", "item_tokens.parquet", "loo_train.parquet", "loo_train_sequences.parquet", "loo_val_sequences.parquet", "loo_val_targets.parquet", "loo_test_targets.parquet", "transformer_vocab_meta_loo.json", "transformer_checkpoint_loo.pkl", "sasrec_checkpoint.pkl", "train.parquet", "val_targets.parquet", "loo_evaluation_results.json")
for p in ["backend", "backend/scripts"]:
    if p not in sys.path: sys.path.insert(0, p)

import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from ipywidgets import interact, widgets
pd.set_option("display.max_colwidth", 90)
P = os.path.join("data", "processed")
print("ready")

![Leave-one-out: hide the last item, rank, score with Recall@K and NDCG@K.](https://raw.githubusercontent.com/juanmigutierrez/generative-recommendation-engine/main/blog/Images/eval_flow_example.png)

*Leave-one-out: hide the last item, rank, score with Recall@K and NDCG@K.*

## 1. One user, two questions

The time split asks: given everything before Nov 2019, what will this user review in 2021–2023? Leave-one-out asks: given everything but the last item, what is the last item? Same person, very different problems.

In [ ]:
import json, pickle, jax, jax.numpy as jnp, time
items = pd.read_parquet(f"{P}/item_catalog.parquet").set_index("item_id"); title = items["description"]
loo_train_seq = {r.user_id: list(r.item_ids) for r in pd.read_parquet(f"{P}/loo_train_sequences.parquet").itertuples()}
loo_val_seq   = {r.user_id: list(r.item_ids) for r in pd.read_parquet(f"{P}/loo_val_sequences.parquet").itertuples()}
loo_val_t  = {r.user_id: set(r.item_ids) for r in pd.read_parquet(f"{P}/loo_val_targets.parquet").itertuples()}
loo_test_t = {r.user_id: set(r.item_ids) for r in pd.read_parquet(f"{P}/loo_test_targets.parquet").itertuples()}
train = pd.read_parquet(f"{P}/train.parquet"); ts_val_t = {r.user_id: set(r.item_ids) for r in pd.read_parquet(f"{P}/val_targets.parquet").itertuples()}
train["timestamp"] = pd.to_datetime(train["timestamp"]); ts_hist = train.sort_values("timestamp").groupby("user_id")["item_id"].apply(list).to_dict()
train_items = set(train.item_id)
rng = np.random.RandomState(3)
both = [u for u in ts_val_t if u in ts_hist and len(ts_hist[u]) >= 3 and len(loo_train_seq[u]) >= 3]
demo_users = sorted(rng.choice(both, 40, replace=False).tolist())

def two_questions(user_id):
    print("TIME SPLIT — context (reviews before Nov 2019):"); [print("   ", title[i][:70]) for i in ts_hist[user_id][-5:]]
    print("  targets (reviews Nov 2019 – Oct 2021):"); [print(f"   {'(never seen in training) ' if i not in train_items else ''}{title[i][:60]}") for i in ts_val_t[user_id]]
    print("\nLEAVE-ONE-OUT — context (all but the last two):"); [print("   ", title[i][:70]) for i in loo_train_seq[user_id][-5:]]
    print("  val target (second-to-last item):"); [print("   ", title[i][:70]) for i in loo_val_t[user_id]]

interact(two_questions, user_id=widgets.Dropdown(options=demo_users, description="user"));

## 2. Load the four models

Popularity and ALS are fit on the leave-one-out training rows in a few seconds; SASRec and the Semantic-ID Transformer come from the checkpoints trained for the post.

In [ ]:
from models.popularity import PopularityRecommender
from models.als import ALSRecommender
from models import sasrec, transformer as tx
import evaluate_retrieval as ev
from models.metrics import recall_at_k, ndcg_at_k

loo_train = pd.read_parquet(f"{P}/loo_train.parquet"); n_items = len(items); n_users = len(pd.read_parquet(f"{P}/user_catalog.parquet"))
pop = PopularityRecommender().fit(loo_train)
t0 = time.time(); als = ALSRecommender(factors=64, regularization=0.05, iterations=15, alpha=40.0).fit(loo_train, n_users, n_items); print(f"ALS fit {time.time()-t0:.0f}s")

sas_raw = pickle.load(open(f"{P}/sasrec_checkpoint.pkl", "rb")); sas_p = jax.tree_util.tree_map(jnp.array, sas_raw["params"]); sas_cfg = sas_raw["config"]
sas_score = jax.jit(lambda t: sasrec.score_batch(sas_p, t, n_items, sas_cfg["n_heads"]))
print(f"SASRec: epoch {sas_raw['epoch']}, {sas_cfg['n_layers']} layers d={sas_cfg['d_model']}, loss {sas_raw['history'][-1]:.2f}")

tf_raw = pickle.load(open(f"{P}/transformer_checkpoint_loo.pkl", "rb")); tf_p = jax.tree_util.tree_map(jnp.array, tf_raw["params"]); tf_cfg = tf_raw["config"]
meta = json.load(open(f"{P}/transformer_vocab_meta_loo.json")); item_tokens = pd.read_parquet(f"{P}/item_tokens.parquet"); trie = ev.build_trie(item_tokens)
item_to_tok = {r.item_id: [r.tok1, r.tok2, r.tok3, r.tok4] for r in item_tokens.itertuples(index=False)}
ev.N_HEADS = tf_cfg["n_heads"]; ev.BEAM_WIDTH = 30
print(f"Semantic-ID Transformer: epoch {tf_raw['epoch']}, {tf_cfg['n_layers']} layers d={tf_cfg['d_model']}, loss {tf_raw['history'][-1]:.2f}")

max_ctx = (meta["max_len"] - 1) // 4 - 1
def tf_context(hist, u): return [meta["item_vocab"] + int(u) % meta["user_buckets"]] + sum((item_to_tok[i] for i in hist[-max_ctx:]), [])

def recommend_all(users, contexts, K=20):
    """contexts: dict user -> item history. Returns {model: {user: ranked list}} with the history filtered out."""
    out = {"popularity": {}, "als": {}, "sasrec": {}, "transformer": {}}
    for u in users:
        seen = set(contexts[u])
        out["popularity"][u] = [i for i in pop.recommend(u, K + 50) if i not in seen][:K]
        out["als"][u] = [i for i in (als.recommend(u, K + 50) if not als.is_cold(u) else pop.recommend(u, K + 50)) if i not in seen][:K]
    for s in range(0, len(users), 256):
        bu = users[s:s+256]; toks = np.full((len(bu), sas_cfg["max_items"]), n_items, dtype=np.int32)
        for k, u in enumerate(bu):
            h = contexts[u][-sas_cfg["max_items"]:]; toks[k, :len(h)] = h
        sc = np.array(sas_score(jnp.array(toks)))
        for k, u in enumerate(bu):
            sc[k, list(contexts[u])] = -np.inf; top = np.argpartition(-sc[k], K)[:K]; out["sasrec"][u] = top[np.argsort(-sc[k][top])].tolist()
    for s in range(0, len(users), 50):
        bu = users[s:s+50]
        for u, ranked in zip(bu, ev.beam_search_batch(tf_p, [tf_context(contexts[u], u) for u in bu], meta, trie)):
            out["transformer"][u] = [i for i, _ in ranked if i not in set(contexts[u])][:K]
    return out

## 3. Four lists for one user

Leave-one-out val: context = all but the last two items, target = the second-to-last.

In [ ]:
def four_lists(user_id, K=10):
    recs = recommend_all([user_id], {user_id: loo_train_seq[user_id]}, K); truth = loo_val_t[user_id]
    print("history:", " | ".join(title[i][:28] for i in loo_train_seq[user_id][-5:])); print("true next item:", title[list(truth)[0]][:80], "\n")
    for name in recs:
        lst = recs[name][user_id]; hit = [r for r, i in enumerate(lst, 1) if i in truth]
        print(f"{name:<12} {'✔ hit at #' + str(hit[0]) if hit else 'miss'}"); [print(f"   {'✔' if i in truth else ' '} {r}. {title[i][:65]}") for r, i in enumerate(lst[:5], 1)]

interact(four_lists, user_id=widgets.Dropdown(options=demo_users, description="user"), K=widgets.IntSlider(10, 5, 20, 5));

## 4. The table, on a sample of users

Recall@K and NDCG@K for K = 5, 10, 20 — the papers' numbers. `N_USERS = 0` evaluates all 94,762 (about 30 min on a T4, mostly beam search). The full-population result from the post is loaded underneath for comparison.

In [ ]:
N_USERS = 3000   #@param {type:"integer"}
SPLIT = "val"    #@param ["val", "test"]
if QUICK: N_USERS = 150
targets_full = loo_val_t if SPLIT == "val" else loo_test_t; contexts = loo_train_seq if SPLIT == "val" else loo_val_seq
users = sorted(targets_full) if N_USERS == 0 else sorted(np.random.RandomState(7).choice(sorted(targets_full), N_USERS, replace=False).tolist())
t0 = time.time(); recs = recommend_all(users, contexts); print(f"{len(users):,} users scored in {time.time()-t0:.0f}s")
rows = {m: {**{f"recall@{k}": np.mean([recall_at_k(recs[m][u], targets_full[u], k) for u in users]) for k in (5, 10, 20)},
            **{f"ndcg@{k}": np.mean([ndcg_at_k(recs[m][u], targets_full[u], k) for u in users]) for k in (5, 10, 20)}} for m in recs}
print(f"\nthis run — {SPLIT}, {len(users):,} users:"); display(pd.DataFrame(rows).T.round(4))
full = json.load(open(f"{P}/loo_evaluation_results.json"))["results"][SPLIT]
print(f"the post — {SPLIT}, all 94,762 users:"); display(pd.DataFrame(full).T.round(4))

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3.2)); names = list(rows); vals = [rows[m]["recall@10"] for m in names]
ax.bar(names, vals, color="#2a78d6", width=0.55); [ax.text(i, v + max(vals) * 0.02, f"{v:.4f}", ha="center", fontsize=9) for i, v in enumerate(vals)]
ax.set_ylabel("recall@10"); ax.set_title(f"leave-one-out {SPLIT}, {len(users):,} users", loc="left"); [ax.spines[s].set_visible(False) for s in ["top", "right"]]
plt.show()
print("Under the papers' protocol the generative model is ~3x popularity and in TIGER's published range — but SASRec, the ID-based Transformer with the same backbone, is still ahead on this dataset. The post's closing section lists the suspects.")

![The post's one-figure summary: the same models under both protocols.](https://raw.githubusercontent.com/juanmigutierrez/generative-recommendation-engine/main/blog/Images/recall_two_protocols.png)

*The post's one-figure summary: the same models under both protocols.*

That is the end of the tutorial series. The full pipeline scripts, docs for each step and the audit that led to the second protocol are in the [repository](https://github.com/juanmigutierrez/generative-recommendation-engine).